# SE-attention + SupCon auxiliary loss to address the precision gap (Correction 3)


Needs GPU. Uses the same fixed 483/104/104 ISIC split (`expA_train/val/test.csv`) and the Stage-1 SSL-pretrained encoder checkpoint as your main result, so it's directly comparable. Toggle `USE_SE_BLOCK` / `USE_SUPCON` to run each intervention alone or combined. Edit the three `EDIT THESE` paths below.

In [2]:
# ============================================================
# Representation-level intervention to close the precision gap
# left open by adaptive class-weighting (Correction 3): adds
# (a) a Squeeze-and-Excitation (SE) attention block after each
#     conv stage of the compact encoder, and/or
# (b) a supervised contrastive (SupCon) auxiliary loss term that
#     pulls MEL embeddings together and pushes them away from
#     BKL/NV in feature space, alongside the existing focal loss.
# Toggle each independently with USE_SE_BLOCK / USE_SUPCON below
# so you can run "SE only", "SupCon only", or "both" as separate
# ablations if you want more than one new row in the paper.
#
# DATASETS NEEDED (edit the paths below):
#   - Pretrained SSL encoder weights (SSL_ENCODER_CKPT), the same
#     Stage-1 checkpoint used in your other fine-tuning notebooks.
#     NOTE: if USE_SE_BLOCK is True, the encoder architecture no
#     longer matches the checkpoint exactly -- the new SE layers
#     are randomly initialised and loaded with strict=False; check
#     the printed "missing/unexpected keys" line to confirm only
#     the new SE layers are unmatched (encoder conv/bn weights
#     should still load correctly).
#   - expA_train.csv / expA_val.csv / expA_test.csv and
#     LABELED_DIR, unchanged -- same fixed 483/104/104 ISIC split
#     as your main result, so this is directly comparable to the
#     existing fixed-weight and adaptive-weight rows.
#
# WHAT THIS DOES:
#   Fine-tunes on the same data/recipe as your main SSL+rebalance
#   result (5x MEL oversampling, heavy MEL augmentation, focal
#   loss gamma=1.5, weights 4/2/1), but the total loss is
#   L = L_focal + LAMBDA_SUPCON * L_supcon (if USE_SUPCON), and/or
#   the encoder includes SE blocks (if USE_SE_BLOCK). Runs 3 seeds.
#
# OUTPUT:
#   - attention_supcon_results.csv: per-seed accuracy, macro-F1,
#     MEL recall, MEL precision, MEL Average Precision -- put this
#     directly alongside the existing fixed-weight/adaptive-weight
#     rows in the paper's main results table and PR-analysis table.
#     The key number to check is whether MEL PRECISION improves
#     relative to the adaptive-weight row (0.172) without giving
#     up as much macro-F1 as adaptive weighting did (0.483).
# ============================================================
import os, glob, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import f1_score, average_precision_score
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- EDIT THESE ----
BACKUP_DIR       = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR      = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
SSL_ENCODER_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'
# ---------------------

USE_SE_BLOCK  = True   # squeeze-and-excitation channel attention in the encoder
USE_SUPCON    = True   # supervised contrastive auxiliary loss on the projector head
LAMBDA_SUPCON = 0.3    # weight on the SupCon term relative to focal loss
SUPCON_TEMP   = 0.1

OUTPUT_DIR = '/kaggle/working'
SEEDS = [42, 123, 2024]
CLASSES = ['BKL', 'MEL', 'NV']
MEL_OVERSAMPLE = 5
MAX_EPOCHS = 30
PATIENCE = 7

train_transform_std = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_transform_mel = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45), transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def resolve_path(img_name):
    if not img_name.endswith('.jpg'):
        img_name += '.jpg'
    p = os.path.join(LABELED_DIR, img_name)
    if not os.path.exists(p):
        m = glob.glob(f"{LABELED_DIR}/**/{img_name}", recursive=True)
        p = m[0] if m else p
    return p


class LesionDataset(Dataset):
    def __init__(self, df, training):
        self.df = df.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(resolve_path(str(row['image']))).convert('RGB')
        tfm = (train_transform_mel if row['label'] == 'MEL' else train_transform_std) if self.training else eval_transform
        return tfm(img), row['label']


def make_oversampled_df(df):
    mel = df[df['label'] == 'MEL']
    rest = df[df['label'] != 'MEL']
    return pd.concat([rest] + [mel] * MEL_OVERSAMPLE, ignore_index=True)


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class ConvBlock(nn.Module):
    """Conv/BN/ReLU/Pool named to match the original encoder's
    parameter shapes (conv, bn), so a Stage-1 checkpoint's weights
    still load into .conv / .bn. Only the optional .se sub-module
    is new and randomly initialised."""
    def __init__(self, in_ch, out_ch, use_se):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.se = SEBlock(out_ch) if use_se else nn.Identity()

    def forward(self, x):
        x = self.pool(self.relu(self.bn(self.conv(x))))
        return self.se(x)


class SimpleEncoder(nn.Module):
    def __init__(self, use_se=False):
        super().__init__()
        self.block1 = ConvBlock(3, 32, use_se)
        self.block2 = ConvBlock(32, 64, use_se)
        self.block3 = ConvBlock(64, 128, use_se)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.flatten(self.gap(x))


class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes, proj_dim=64):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
        self.projector = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, proj_dim))

    def forward(self, x, return_proj=False):
        feat = self.encoder(x)
        logits = self.classifier(feat)
        if return_proj:
            proj = F.normalize(self.projector(feat), dim=1)
            return logits, proj
        return logits


def load_legacy_encoder_weights(encoder, ckpt_path):
    """The original checkpoints use flat keys like
    'features.0/1/4/5/8/9.*' (optionally prefixed with 'encoder.'
    if saved from the full SSLClassifier). Remap those onto the
    new block1/2/3.conv/bn.* structure. SE layers are new and are
    left at their random init (strict=False)."""
    state = torch.load(ckpt_path, map_location=device)
    stripped = {(k[len('encoder.'):] if k.startswith('encoder.') else k): v for k, v in state.items()}
    remap = {
        'features.0': 'block1.conv', 'features.1': 'block1.bn',
        'features.4': 'block2.conv', 'features.5': 'block2.bn',
        'features.8': 'block3.conv', 'features.9': 'block3.bn',
    }
    new_state = {}
    for k, v in stripped.items():
        for old_prefix, new_prefix in remap.items():
            if k.startswith(old_prefix + '.'):
                new_state[new_prefix + k[len(old_prefix):]] = v
                break
    missing, unexpected = encoder.load_state_dict(new_state, strict=False)
    print(f"Encoder load: {len(missing)} missing keys (expected: SE layers if USE_SE_BLOCK), "
          f"{len(unexpected)} unexpected keys.")
    return encoder


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=1.5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        a_t = self.alpha[targets]
        return (-a_t * (1 - p_t) ** self.gamma * logp_t).mean()


class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al., 2020), single-view version."""
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        n = features.shape[0]
        sim = torch.matmul(features, features.T) / self.temperature
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()
        mask_self = torch.eye(n, device=features.device).bool()
        labels = labels.view(-1, 1)
        pos_mask = (labels == labels.T) & ~mask_self
        exp_sim = torch.exp(sim) * (~mask_self)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)
        pos_counts = pos_mask.sum(dim=1)
        valid = pos_counts > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=features.device)
        loss = -(log_prob * pos_mask).sum(dim=1)[valid] / pos_counts[valid]
        return loss.mean()


def evaluate(model, loader):
    model.eval()
    all_true, all_pred, mel_scores = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_pred.extend([CLASSES[p] for p in preds])
            all_true.extend(labels)
            mel_scores.extend(probs[:, CLASSES.index('MEL')].cpu().numpy())
    true_arr, pred_arr = np.array(all_true), np.array(all_pred)
    acc = np.mean(pred_arr == true_arr)
    macro_f1 = f1_score(true_arr, pred_arr, labels=CLASSES, average='macro', zero_division=0)
    tp = np.sum((true_arr == 'MEL') & (pred_arr == 'MEL'))
    fn = np.sum((true_arr == 'MEL') & (pred_arr != 'MEL'))
    fp = np.sum((true_arr != 'MEL') & (pred_arr == 'MEL'))
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    ap = average_precision_score((true_arr == 'MEL').astype(int), mel_scores)
    return acc, macro_f1, recall, precision, ap


class_to_idx = {c: i for i, c in enumerate(CLASSES)}
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))

results = []
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(LesionDataset(make_oversampled_df(train_df), training=True), batch_size=16, shuffle=True)
    val_loader = DataLoader(LesionDataset(val_df, training=False), batch_size=16, shuffle=False)
    test_loader = DataLoader(LesionDataset(test_df, training=False), batch_size=16, shuffle=False)

    encoder = SimpleEncoder(use_se=USE_SE_BLOCK)
    encoder = load_legacy_encoder_weights(encoder, SSL_ENCODER_CKPT)
    model = SSLClassifier(encoder, len(CLASSES)).to(device)

    alpha = torch.tensor([2.0, 4.0, 1.0], device=device)
    focal = FocalLoss(alpha, gamma=1.5)
    supcon = SupConLoss(temperature=SUPCON_TEMP)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            targets = torch.tensor([class_to_idx[l] for l in labels], device=device)
            optimizer.zero_grad()
            if USE_SUPCON:
                logits, proj = model(images, return_proj=True)
                loss = focal(logits, targets) + LAMBDA_SUPCON * supcon(proj, targets)
            else:
                logits = model(images)
                loss = focal(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        _, val_f1, _, _, _ = evaluate(model, val_loader)
        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    acc, f1_, rec, prec, ap = evaluate(model, test_loader)
    print(f"[seed {seed}] acc={acc:.3f} macroF1={f1_:.3f} MELrec={rec:.3f} MELprec={prec:.3f} MEL_AP={ap:.3f}")
    results.append({'seed': seed, 'accuracy': acc, 'macro_f1': f1_,
                     'mel_recall': rec, 'mel_precision': prec, 'mel_ap': ap})

results_df = pd.DataFrame(results)
results_df.to_csv(f'{OUTPUT_DIR}/attention_supcon_results.csv', index=False)
print(f"\nWrote {OUTPUT_DIR}/attention_supcon_results.csv")
print(f"Config: USE_SE_BLOCK={USE_SE_BLOCK}, USE_SUPCON={USE_SUPCON}, LAMBDA_SUPCON={LAMBDA_SUPCON}")
print("\nMean +/- SD across 3 seeds (compare mel_precision to adaptive-weight's 0.172):")
for col in ['accuracy', 'macro_f1', 'mel_recall', 'mel_precision', 'mel_ap']:
    print(f"  {col:15s}: {results_df[col].mean():.3f} +/- {results_df[col].std():.3f}")


Encoder load: 12 missing keys (expected: SE layers if USE_SE_BLOCK), 0 unexpected keys.
[seed 42] acc=0.625 macroF1=0.484 MELrec=0.375 MELprec=0.158 MEL_AP=0.237
Encoder load: 12 missing keys (expected: SE layers if USE_SE_BLOCK), 0 unexpected keys.
[seed 123] acc=0.683 macroF1=0.516 MELrec=0.375 MELprec=0.231 MEL_AP=0.267
Encoder load: 12 missing keys (expected: SE layers if USE_SE_BLOCK), 0 unexpected keys.
[seed 2024] acc=0.702 macroF1=0.549 MELrec=0.375 MELprec=0.200 MEL_AP=0.324

Wrote /kaggle/working/attention_supcon_results.csv
Config: USE_SE_BLOCK=True, USE_SUPCON=True, LAMBDA_SUPCON=0.3

Mean +/- SD across 3 seeds (compare mel_precision to adaptive-weight's 0.172):
  accuracy       : 0.670 +/- 0.040
  macro_f1       : 0.516 +/- 0.032
  mel_recall     : 0.375 +/- 0.000
  mel_precision  : 0.196 +/- 0.037
  mel_ap         : 0.276 +/- 0.044
